### Setup library

In [7]:
!pip install -q unsloth trl accelerate bitsandbytes datasets pandas

In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Process Dataset

In [11]:
import pandas as pd

df = pd.read_csv("/content/test_final.csv")
df = df.dropna(subset=["prompt", "essay", "band"])
df["band"] = df["band"].astype(str).str.strip().replace({"<4": "3.5"}).astype(float)

In [12]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner. Evaluate the essay using IELTS band descriptors.
Internally calculate scores for the four criteria, average them, and round to the nearest 0.5.
Return ONLY the overall band score in strict JSON format.

### Input:
Essay prompt: {}
Essay: {}

### Response:
{}"""

### Format to prompt response

In [13]:
import json

EOS_TOKEN = tokenizer.eos_token

def format_band_to_json(band_value):
    return json.dumps({"overallScore": band_value})

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays  = examples["essay"]
    outputs = examples["band"]
    texts = []
    for prompt, essay, output in zip(prompts, essays, outputs):
        output_json = format_band_to_json(output)
        text = alpaca_prompt.format(prompt, essay, output_json) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }


### Train and crteate DataSet

In [14]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["prompt", "essay", "band"]])
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

Map:   0%|          | 0/495 [00:00<?, ? examples/s]

### LoRA

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
    use_rslora = False,
)


Unsloth 2025.10.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/content/mistral_band_model",
        report_to = "none",
        eval_strategy = "steps",
        eval_steps = 100,
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/445 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

### Train

In [17]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 445 | Num Epochs = 3 | Total steps = 168
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.490900,1.718958


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=168, training_loss=1.5592851014364333, metrics={'train_runtime': 2344.7288, 'train_samples_per_second': 0.569, 'train_steps_per_second': 0.072, 'total_flos': 3.306626971963392e+16, 'train_loss': 1.5592851014364333, 'epoch': 3.0})

### Loss func

In [18]:
from transformers import TextStreamer
import json
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

streamer = TextStreamer(tokenizer)
predicted_scores = []
true_scores = []

for example in eval_dataset:
    # Tạo prompt giống lúc huấn luyện
    prompt = alpaca_prompt.format(example["prompt"], example["essay"], "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Sinh văn bản từ mô hình
    output = model.generate(**inputs, max_new_tokens=128)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Trích xuất band điểm từ JSON
    try:
        json_str = decoded.split("### Response:")[-1].strip()
        band = json.loads(json_str)["overallScore"]
        predicted_scores.append(float(band))
        true_scores.append(float(example["band"]))
    except:
        continue  # bỏ qua nếu lỗi định dạng


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score
import numpy as np

mae = mean_absolute_error(true_scores, predicted_scores)
mse = mean_squared_error(true_scores, predicted_scores)
rmse = np.sqrt(mse)

# Ép về string để sklearn coi là nhãn discrete
y_pred_class = y_pred_class.astype(str)
y_true_class = y_true_class.astype(str)

acc = accuracy_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class, average='weighted')
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE: 1.12
RMSE: 1.41
